### Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import scipy.io
import matplotlib.gridspec as gridspec
import urllib.request
import os
import math

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

In [ ]:
k              = 0.05
epochs         = 5
batch_size     = 16
bottleneck_dim = 25
lr             = 1e-3
# momentum       = 0.9
dataset        = "cifar10_patches"

### Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)
IMAGES_MAT_PATH = '../data/IMAGES.mat'
if not os.path.exists(IMAGES_MAT_PATH):
    url = 'https://raw.githubusercontent.com/fanfeng2015/Sparse-Autoencoder/master/IMAGES.mat'
    urllib.request.urlretrieve(url, IMAGES_MAT_PATH)

In [ ]:
class AndrewNgLandscape(Dataset):
    def __init__(self):
        mat = scipy.io.loadmat(IMAGES_MAT_PATH)

        self.wh_landscapes = torch.Tensor(mat['IMAGES']).permute(2, 0, 1)  # (num_images, H, W)
        self.num_samples   = 10000
        self.patch_size    = 8
        self.image_height  = self.wh_landscapes.shape[1]
        self.image_width   = self.wh_landscapes.shape[2]

        self.grid_h = self.image_height // self.patch_size
        self.grid_w = self.image_width // self.patch_size

        self.patches = []
        for _ in range(self.num_samples):
            image_idx = torch.randint(0, self.wh_landscapes.shape[0], (1,)).item()
            grid_x    = torch.randint(0, self.grid_h, (1,)).item()
            grid_y    = torch.randint(0, self.grid_w, (1,)).item()

            x_start = grid_x * self.patch_size
            y_start = grid_y * self.patch_size

            patch = self.wh_landscapes[
                image_idx,
                x_start : x_start + self.patch_size,
                y_start : y_start + self.patch_size,
            ]
            self.patches.append(patch.flatten())

        self.patches = torch.stack(self.patches)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.patches[idx], 0

In [ ]:
class MNISTPatches(Dataset):
    def __init__(self):
        raw               = datasets.MNIST('../data', train=True, download=True,
                                           transform=transforms.ToTensor())
        loader            = DataLoader(raw, batch_size=len(raw), shuffle=False)
        images, _         = next(iter(loader))
        self.wh_mnist      = images[:, 0]       # Shape: (num_images, height, width)
        self.num_samples   = 10000
        self.patch_size    = 8
        self.image_height  = self.wh_mnist.shape[1]
        self.image_width   = self.wh_mnist.shape[2]

        self.grid_h = self.image_height // self.patch_size
        self.grid_w = self.image_width // self.patch_size

        self.patches = []
        for _ in range(self.num_samples):
            image_idx = torch.randint(0, self.wh_mnist.shape[0], (1,)).item()
            grid_x    = torch.randint(0, self.grid_h, (1,)).item()
            grid_y    = torch.randint(0, self.grid_w, (1,)).item()

            x_start = grid_x * self.patch_size
            y_start = grid_y * self.patch_size

            patch = self.wh_mnist[
                image_idx,
                x_start : x_start + self.patch_size,
                y_start : y_start + self.patch_size,
            ]
            self.patches.append(patch.flatten())

        self.patches = torch.stack(self.patches)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.patches[idx], 0

In [ ]:
class CIFAR10Patches(Dataset):
    def __init__(self):
        raw                = datasets.CIFAR10('../data', train=True, download=True,
                                             transform=transforms.Compose([
                                                 transforms.Grayscale(num_output_channels=1),
                                                 transforms.ToTensor(),
                                             ]))
        loader             = DataLoader(raw, batch_size=len(raw), shuffle=False)
        images, _          = next(iter(loader))
        self.wh_cifar10    = images[:, 0]       # Shape: (num_images, height, width)
        self.num_samples   = 10000
        self.patch_size    = 8
        self.image_height  = self.wh_cifar10.shape[1]
        self.image_width   = self.wh_cifar10.shape[2]

        self.grid_h = self.image_height // self.patch_size
        self.grid_w = self.image_width // self.patch_size

        self.patches = []
        for _ in range(self.num_samples):
            image_idx = torch.randint(0, self.wh_cifar10.shape[0], (1,)).item()
            grid_x    = torch.randint(0, self.grid_h, (1,)).item()
            grid_y    = torch.randint(0, self.grid_w, (1,)).item()

            x_start = grid_x * self.patch_size
            y_start = grid_y * self.patch_size

            patch = self.wh_cifar10[
                image_idx,
                x_start : x_start + self.patch_size,
                y_start : y_start + self.patch_size,
            ]
            self.patches.append(patch.flatten())

        self.patches = torch.stack(self.patches)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.patches[idx], 0

In [ ]:
def get_data_loader(dataset: str) -> DataLoader:
    if dataset == "landscapes_patches":
        return DataLoader(AndrewNgLandscape(), 
                          batch_size=batch_size, shuffle=True, pin_memory=True)

    elif dataset == "mnist_patches":
        return DataLoader(MNISTPatches(), 
                          batch_size=batch_size, shuffle=True, pin_memory=True)

    elif dataset == "cifar10_patches":
        return DataLoader(CIFAR10Patches(), 
                          batch_size=batch_size, shuffle=True, pin_memory=True)

    elif dataset == "mnist":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
            lambda x: x.flatten(),
        ])
        return DataLoader(datasets.MNIST('../data', train=True, download=True, transform=transform),
                          batch_size=batch_size, shuffle=True, pin_memory=True)

    elif dataset == "cifar10":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Grayscale(num_output_channels=1),
            transforms.Normalize((0.5,), (0.5,)),
            lambda x: x.flatten(),
        ])
        return DataLoader(datasets.CIFAR10('../data', train=True, download=True, transform=transform),
                          batch_size=batch_size, shuffle=True, pin_memory=True)

    else:
        raise ValueError(f"Unknown dataset: {dataset!r}")

In [ ]:
def get_flattened_size(dataset: str) -> int:
    if dataset in ("landscapes_patches", "mnist_patches", "cifar10_patches"):
        return 8 * 8

    elif dataset == "mnist":
        return 28 * 28

    elif dataset == "cifar10":
        return 32 * 32

    else:
        raise ValueError(f"Unknown dataset: {dataset!r}")

In [ ]:
data_loader = get_data_loader(dataset)

### Autoencoders

In [ ]:
class Autoencoder(nn.Module):
    def __init__(
        self,
        dim: tuple
    ) -> None:
        super().__init__()

        self.input_dim, self.bottleneck_dim = dim

        self.encoder = nn.Linear(self.input_dim, self.bottleneck_dim)
        self.sigmoid = nn.Sigmoid()
        self.decoder = nn.Linear(self.bottleneck_dim, self.input_dim)

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:
        if x.ndim == 1:
            x = x.unsqueeze(0)

        z1 = self.encoder(x)
        a1 = self.sigmoid(z1)
        z2 = self.decoder(a1)

        return z2

In [ ]:
class K_Sparse_Autoencoder(nn.Module):
    def __init__(
        self,
        dim: tuple,
        a: int,
        k: float,
        total_epochs: int,
    ) -> None:
        super().__init__()

        self.input_dim, self.bottleneck_dim = dim
        self.a            = a
        self.k            = k
        self.k_start      = 1.0
        self.total_epochs = total_epochs

        self.encoder      = nn.Linear(self.input_dim, self.bottleneck_dim)
        self.identity     = nn.Identity()
        self.decoder_bias = nn.Parameter(torch.zeros(self.input_dim))

    def forward(
        self,
        x: torch.Tensor,
        epoch: int = 0,
    ) -> torch.Tensor:
        if x.ndim == 1:
            x = x.unsqueeze(0)

        z1 = self.encoder(x)
        a1 = self.identity(z1)

        # layer sparsity
        if self.training:
            anneal_epochs = self.total_epochs // 2
            if epoch < anneal_epochs:
                progress  = epoch / anneal_epochs
                current_k = self.k_start + progress * (self.k - self.k_start)
            else:
                current_k = self.k
        else:
            current_k = self.a * self.k

        _, a1_topk_idx = torch.topk(a1, max(1, int(self.bottleneck_dim * current_k)), dim=1)
        a1_wta_mask = torch.zeros_like(a1)
        a1_wta_mask.scatter_(1, a1_topk_idx, 1)
        a1 = a1 * a1_wta_mask

        z2 = F.linear(a1, self.encoder.weight, self.decoder_bias)

        return z2

In [ ]:
class FC_WTA_Autoencoder(nn.Module):
    def __init__(
        self,
        dim: tuple,
        k: float,
    ) -> None:
        super().__init__()

        self.input_dim, self.bottleneck_dim = dim
        self.k = k

        self.encoder      = nn.Linear(self.input_dim, self.bottleneck_dim)
        self.relu         = nn.ReLU()
        self.decoder_bias = nn.Parameter(torch.zeros(self.input_dim))

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:
        if x.ndim == 1:
            x = x.unsqueeze(0)

        z1 = self.encoder(x)
        a1 = self.relu(z1)

        # batch sparsity
        if (self.training):
            _, a1_topk_idx = torch.topk(a1, max(1, int(x.shape[0] * self.k)), dim=0)
            a1_wta_mask = torch.zeros_like(a1)
            a1_wta_mask.scatter_(0, a1_topk_idx, 1)
            a1 = a1 * a1_wta_mask
            
        z2 = F.linear(a1, self.encoder.weight, self.decoder_bias)

        return z2

In [ ]:
input_dim = get_flattened_size(dataset)

normal_autoencoder = Autoencoder(dim=(input_dim, bottleneck_dim)).to(device)
normal_criterion   = nn.MSELoss()
normal_optimizer   = torch.optim.Adam(normal_autoencoder.parameters(), lr=lr)

sparse_autoencoder = K_Sparse_Autoencoder(dim=(input_dim, bottleneck_dim), k=k, total_epochs=epochs, a=1).to(device)
sparse_criterion   = nn.MSELoss()
sparse_optimizer   = torch.optim.Adam(sparse_autoencoder.parameters(), lr=lr)

fc_wta_autoencoder = FC_WTA_Autoencoder(dim=(input_dim, bottleneck_dim), k=k).to(device)
fc_wta_criterion   = nn.MSELoss()
fc_wta_optimizer   = torch.optim.Adam(fc_wta_autoencoder.parameters(), lr=lr)

### Training

In [ ]:
normal_losses = []
sparse_losses = []
fc_wta_losses = []

normal_autoencoder.train()
sparse_autoencoder.train()
fc_wta_autoencoder.train()
epochbar = tqdm(range(1, epochs + 1), desc="Epochs")

for epoch in epochbar:
    normal_epoch_loss = 0.0
    sparse_epoch_loss = 0.0
    fc_wta_epoch_loss = 0.0
    for batch_inputs, _ in data_loader:
        batch_inputs = batch_inputs.to(device)
        normal_optimizer.zero_grad()
        sparse_optimizer.zero_grad()
        fc_wta_optimizer.zero_grad()

        normal_x_recon = normal_autoencoder(batch_inputs)
        sparse_x_recon = sparse_autoencoder(batch_inputs, epoch=epoch)
        fc_wta_x_recon = fc_wta_autoencoder(batch_inputs)

        normal_loss = normal_criterion(normal_x_recon, batch_inputs)
        sparse_loss = sparse_criterion(sparse_x_recon, batch_inputs, epoch=epoch)
        fc_wta_loss = fc_wta_criterion(fc_wta_x_recon, batch_inputs)

        normal_loss.backward()
        sparse_loss.backward()
        fc_wta_loss.backward()

        normal_optimizer.step()
        sparse_optimizer.step()
        fc_wta_optimizer.step()

        normal_batch_loss = normal_loss.item()
        sparse_batch_loss = sparse_loss.item()
        fc_wta_batch_loss = fc_wta_loss.item()

        normal_epoch_loss += normal_batch_loss * batch_inputs.size(0)
        sparse_epoch_loss += sparse_batch_loss * batch_inputs.size(0)
        fc_wta_epoch_loss += fc_wta_batch_loss * batch_inputs.size(0)

        epochbar.set_postfix({
            "normal_loss": f"{normal_batch_loss:.4f}",
            "sparse_loss": f"{sparse_batch_loss:.4f}",
            "fc_wta_loss": f"{fc_wta_batch_loss:.4f}"
        })

    normal_epoch_loss /= len(data_loader.dataset)
    sparse_epoch_loss /= len(data_loader.dataset)
    fc_wta_epoch_loss /= len(data_loader.dataset)

    normal_losses.append(normal_epoch_loss)
    sparse_losses.append(sparse_epoch_loss)
    fc_wta_losses.append(fc_wta_epoch_loss)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(range(1, epochs + 1), normal_losses, "o-")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean Train Loss")
axes[0].set_title("Normal Autoencoder Training Performance")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, epochs + 1), sparse_losses, "o-", color='orange')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean Train Loss")
axes[1].set_title("Sparse Autoencoder Training Performance")
axes[1].grid(True, alpha=0.3)

axes[2].plot(range(1, epochs + 1), fc_wta_losses, "o-", color='green')
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Mean Train Loss")
axes[2].set_title("FC-WTA Autoencoder Training Performance")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualizations

In [ ]:
def maximal_activation_normalize(W):
    return W / torch.sqrt(torch.sum(W**2, dim=1, keepdim=True) + 1e-8)

W_enc_normal = maximal_activation_normalize(normal_autoencoder.encoder.weight.detach().cpu())
W_enc_sparse = maximal_activation_normalize(sparse_autoencoder.encoder.weight.detach().cpu())
W_enc_fc_wta = maximal_activation_normalize(fc_wta_autoencoder.encoder.weight.detach().cpu())

kernels_normal = W_enc_normal.reshape(-1, 1, int(math.sqrt(input_dim)), int(math.sqrt(input_dim)))
kernels_sparse = W_enc_sparse.reshape(-1, 1, int(math.sqrt(input_dim)), int(math.sqrt(input_dim)))
kernels_fc_wta = W_enc_fc_wta.reshape(-1, 1, int(math.sqrt(input_dim)), int(math.sqrt(input_dim)))

grid_normal = make_grid(kernels_normal, nrow=5, normalize=True, scale_each=True, padding=1)
grid_sparse = make_grid(kernels_sparse, nrow=5, normalize=True, scale_each=True, padding=1)
grid_fc_wta = make_grid(kernels_fc_wta, nrow=5, normalize=True, scale_each=True, padding=1)

fig = plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
plt.imshow(grid_normal.permute(1, 2, 0))
plt.axis('off')
plt.title('Normal Autoencoder Weights')

plt.subplot(1, 3, 2)
plt.imshow(grid_sparse.permute(1, 2, 0))
plt.axis('off')
plt.title('Sparse Autoencoder Weights')

plt.subplot(1, 3, 3)
plt.imshow(grid_fc_wta.permute(1, 2, 0))
plt.axis('off')
plt.title('FC-WTA Autoencoder Weights')

plt.tight_layout()
plt.show()

In [ ]:
normal_autoencoder.eval()
sparse_autoencoder.eval()
fc_wta_autoencoder.eval()

batch_inputs, _ = next(iter(data_loader))
batch_inputs = batch_inputs.to(device)

with torch.no_grad():
    act_normal = torch.sigmoid(normal_autoencoder.encoder(batch_inputs))

    act_sparse_raw = sparse_autoencoder.encoder(batch_inputs)
    k_eval = max(1, int(sparse_autoencoder.bottleneck_dim * sparse_autoencoder.a * sparse_autoencoder.k))
    _, sparse_topk_idx = torch.topk(act_sparse_raw, k_eval, dim=1)
    sparse_mask = torch.zeros_like(act_sparse_raw).scatter_(1, sparse_topk_idx, 1)
    act_sparse = act_sparse_raw * sparse_mask

    act_fc_wta = fc_wta_autoencoder.relu(fc_wta_autoencoder.encoder(batch_inputs))

sample_idx = torch.randint(0, batch_inputs.shape[0], (1,)).item()

W_enc_normal = normal_autoencoder.encoder.weight.detach().cpu()
W_enc_sparse = sparse_autoencoder.encoder.weight.detach().cpu()
W_enc_fc_wta = fc_wta_autoencoder.encoder.weight.detach().cpu()

a_normal = act_normal[sample_idx].cpu().unsqueeze(1)  # (bottleneck_dim, 1)
a_sparse = act_sparse[sample_idx].cpu().unsqueeze(1)
a_fc_wta = act_fc_wta[sample_idx].cpu().unsqueeze(1)

act_kernels_normal = (a_normal * W_enc_normal).reshape(-1, 1, int(math.sqrt(input_dim)), int(math.sqrt(input_dim)))
act_kernels_sparse = (a_sparse * W_enc_sparse).reshape(-1, 1, int(math.sqrt(input_dim)), int(math.sqrt(input_dim)))
act_kernels_fc_wta = (a_fc_wta * W_enc_fc_wta).reshape(-1, 1, int(math.sqrt(input_dim)), int(math.sqrt(input_dim)))

grid_act_normal = make_grid(act_kernels_normal, nrow=5, normalize=True, scale_each=True, padding=1)
grid_act_sparse = make_grid(act_kernels_sparse, nrow=5, normalize=True, scale_each=True, padding=1)
grid_act_fc_wta = make_grid(act_kernels_fc_wta, nrow=5, normalize=True, scale_each=True, padding=1)

fig = plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
plt.imshow(grid_act_normal.permute(1, 2, 0))
plt.axis('off')
plt.title(f'Normal AE Activations (sample {sample_idx})')

plt.subplot(1, 3, 2)
plt.imshow(grid_act_sparse.permute(1, 2, 0))
plt.axis('off')
plt.title(f'K-Sparse AE Activations (sample {sample_idx})')

plt.subplot(1, 3, 3)
plt.imshow(grid_act_fc_wta.permute(1, 2, 0))
plt.axis('off')
plt.title(f'FC-WTA AE Activations (sample {sample_idx})')

plt.tight_layout()

save_dir = '../results/sparse_autoencoder'
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f'activations_{dataset}_sample{sample_idx}.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'[saved] {save_path}')

plt.show()